# SoftMax Optimizations 

This notebook will go over implementing a revamped version of the `Softmax.backward(self, dvalues)` method.

In [ ]:
try:
    import cupy as cp
    xp = cp
except (ModuleNotFoundError, ImportError):
    import numpy as np
    xp = np 

class SoftMax:
    def forward(self, inputs, training):
        self.exp_values = xp.exp(inputs - xp.max(inputs, axis=1, keepdims = True)) #e**(inputs - max(inputs by row))
        probabilities = self.exp_values / xp.sum(self.exp_values, axis=1, keepdims = True) #e**k / sum(e**k) 
        self.output = probabilities

        return self.output

    # For this notebook we'll ignore this, as we normally don't use this backwards pass
    # anyways assuming the user is using a CCE loss
    def backward(self, dvalues): 
        self.dinputs = xp.empty_like(dvalues) 

        for index, (single_output, single_dvalues) in \
            enumerate(zip(self.output, dvalues)): 
            #Flatten output array 
            single_output = single_output.reshape(-1, 1) 
            #Jacobian matrix
            jacobian = xp.diagflat(single_output) - \
                       xp.dot(single_output, single_output.T)
            #Get sample-wise gradient 
            self.dinputs[index] = xp.dot(jacobian, single_dvalues)     

## Why are we not Trying to Improve `SoftMax.forward()`? 

Softmax will use the equation below to calculate probabilities from a given input tensor for a loss function to interpret. Since we deal with large tensors, its reasonable to perform the Softmax function with respect to each row, rather then each individual element: 

$$
\text{Softmax}(x^{(n)})_i = \frac{\text{exp}(x_{i}^{(n)} - \text{max}_j x_{j}^{(n)})}{\sum _k \text{exp}(x_{k}^{(n)}-\text{max}_j x_j^{(n)})}
$$

where:

* $n$: The batch of samples 
* $i$: The class whose probabilitiy we're computing
* $j$: Used ot find the maximum logit in the current row
* $k$: Sums over ALL classes in the denominator 

Given the equation, it is a trivial task to create a forward method for the Softmax layer. However, we observe again that there are some out of place operations, along with non-reduction operations (such as cp.exp) being used. This means, we'd be unable to combine multiple operations into a fused kernel `@cp.fuse()`. Other deep learning frameworks such as PyTorch or TensorFlow write custom kernels that circumvent the row reduction and elementwise operations into a single kernel. In our case, we'll still have 5 kernel launches which we are unable to avoid unless we create `cp.RawKernel()` method for both HIP and CUDA architectures. 

We still can make a minor optimization, as don't have to keep a `exp_values`, meaning we can omit `self.` from the variable. This reduces our VRAM footprint and allows us to solve the second portion of code slightly faster. Even for the case of combining the backward passes of CE loss and Softmax, we still do not rely on `exp_values`. 

## How can we Improve `SoftMax.backward()`?  

Normally, a framework will account for a user needing a Softmax output activation function along with a CE loss function. When these two are combined, the combined derivative becomes:

$$\frac{\partial L}{\partial z_j} = a_j - y_j$$
where:
$j$: A specific output class node or class neuron in the output layer. 
$a$: The forward-pass predicted probability `self.output`
$y$: The ground truth label, think `y_true`

This approach is blazingly fast, as we only have to perform one operation. If we include batches, then the formula divides the answer by the batch size. 

In the event that a user still wants to use `SoftMax.backward()` directly, its important to still update the pass as it has glaring issues that prevent the code from being vectorized. The isolated analytical formula for `SoftMax.backward()` is
$$
\frac{\partial L}{\partial z_j} = a_j \left( \frac{\partial L}{\partial a_j} - \sum_{i} \frac{\partial L}{\partial a_i} a_i \right)
$$
where: 

* $\frac{\partial L}{\partial z_j}$: Our `dinputs`, the loss with respect to a single output neuron $z$ at the $j$ neuron. 
* $a_j$: Our `self.output`, the actual probability of the $j^\text{{th}}$ output being selected. 
* $\frac{\partial L}{\partial a_j}$: Our `dvalues`, but only at the $j^\text{{th}}$ position or neuron. 
* $\sum_{i} \frac{\partial L}{\partial a_i} a_i$: Our `sum_dvalues_output`. While not defined yet, this is the sum-product of all upstream loss gradients `dvalues` multiplied by their respective Softmax output probabilities across all class features for that sample. 

In a simple example

```python
import numpy as np 
# our softmax probabilities, which WILL add up to 1
output = np.array([0.5, 0.25, 0.25])

dvalues = np.array([2.0, -1.0, 4.0])
#  Perform element wise multiplication (by row)
sum_dvalues_output = cp.sum(dvalues*output, axis = -1, keepdims=True)
```
This creates a tensor of the form 
$$[2.0 \times 0.5,\quad -1.0 \times 0.25,\quad 4.0 \times 0.25]$$
$$= [1.0,\quad -0.25,\quad 1.0]$$

Now we can tackle the next part of the kernel, `cp.sum()`. Since we have to sum the probabilities of each neuron, we must sum by the row, but since we must account for 4D tensors, we have to sum by the channel C. This means we use `axis = -1`. the output for `sum_dvalues_output` becomes:

$$
1.0 + (-0.25) + 1.0 = 1.75 \quad \text{sum dvalues output shape becomes (1, 1)}
$$

From our original equation, we already have $a_j$ in the form of `self.output`, and we already have $\frac{\partial L}{\partial a_j}$ as this is our `dvalues`. meaning our final calculation becomes. 

```python
self.dinputs = self.output * (dvalues - sum_dvalues_output)
# Our example becomes
```

$$
[0.5, 0.25, 0.25] \cdot [2.0 - 1.75, \quad (-1.0) - 1.75,\quad  4.0 - 1.75]
$$
$$
[0.5, 0.25, 0.25] \cdot [0.25, \quad -2.75, \quad 2.25]
$$
We can perform elementwise multiplication to arrive at our output
$$
[0.5 \times 0.25, \quad 0.25 \times (-2.75), \quad 0.25 \times 2.25]
$$
$$
= [0.125, \quad -0.6875, \quad 0.5625]
$$

We have know shown an implementation of the Softmax downstream gradient, along with an example that we could follow along with. The final code for the method is below. 

In [ ]:
class SoftMax:
    def forward(self, inputs, training):
        # we remove the unecessary self. from exp_values 
        exp_values = xp.exp(inputs - xp.max(inputs, axis=1, keepdims = True)) #e**(inputs - max(inputs by row))
        probabilities = self.exp_values / xp.sum(exp_values, axis=1, keepdims = True) #e**k / sum(e**k) 
        self.output = probabilities

        return self.output

    # A vectorized pass of the SoftMax backwards pass
    def backward(self, dvalues): 

        sum_dvalues_output = xp.sum(dvalues * self.output, axis = -1, keepdims=True)
        self.dinputs = self.output * (dvalues - sum_dvalues_output)